# Project 2: routing wafers with local defects to the right root-cause team

**DBA5106, Project 2, Version 2 (with data).** Dataset: WM-811K wafer maps (Kaggle `qingyi/wm811k-wafer-map`).

## The decision this model supports
A fab's yield team reviews wafers that show a **local cluster of failing dies**. There are two look-alike patterns with different root causes:

| Pattern | Typical root cause | What the fab does |
|---|---|---|
| **Edge-Loc** (positive class) | A systematic edge-process problem on a tool (edge-ring wear, edge-bead removal, clamp or handling contact). It repeats on every wafer the tool processes. | Hold the tool and inspect its edge hardware. |
| **Loc** (negative class) | A particle or local event, usually specific to that wafer. | Log it, and review the wafer in the normal queue. |

The model reads a wafer map and predicts **P(Edge-Loc)**. If the prediction is "Edge-Loc", the tool is held for inspection.

## Why the evaluation loss is not the misclassification rate
The two mistakes don't cost the same:
- **False negative** (an Edge-Loc wafer called Loc): the tool keeps running with the edge problem until the next routine monitor catches it, so more lots lose their edge dies.
- **False positive** (a Loc wafer called Edge-Loc): the tool is held for an inspection that finds nothing. That costs some tool time, and nothing else.

We assume a missed edge excursion costs **5 times** an unnecessary tool hold. This ratio is our assumption (the dataset has no cost data), so section 8 shows how the decision changes for ratios from 1 to 10.

$$\text{evaluation loss} = \frac{5 \cdot FN + 1 \cdot FP}{n} \quad \text{(cost per wafer, in units of one unnecessary tool hold)}$$

## How each requirement is met
| Requirement | Where |
|---|---|
| Classification; evaluation loss deviates from misclassification, from the context | Above, and section 3 |
| At least one glassbox and one blackbox | Section 4: L1-penalised linear model; gradient-boosted trees |
| Each model in two versions: L2 loss vs another surrogate loss, tuned separately | Section 4: squared error vs binomial deviance (log loss), four separate searches |
| Compare all of them on validation AUC (not train loss) and decide | Section 5 |
| Threshold that minimises the evaluation loss, for the chosen model | Section 6 |
| Is the model calibrated? If not, improve it | Section 7 |
| Tree SHAP on a tree-based blackbox, local and global | Section 9 |
| Good habits: few explorations, no validation overfitting | Sections 2, 4 and 10 |

## What we did differently
1. **Our own features.** Most public work on this dataset feeds the images to a CNN. We turn each map into 10 physical features, so a glassbox can compete and SHAP has something to say.
2. **A harder question.** "Defect vs no defect" is almost perfectly predictable on this data (AUC 0.995 in our screening). We changed the question to one a fab needs answered and that isn't trivial.
3. **Lot-aware splits.** Wafers from one lot look alike. We keep every lot inside a single split, so the model is never validated on a lot it trained on.
4. **Three separate jobs for the data.** Hyperparameters are tuned by cross-validation inside the training split. The validation split is used only to compare the four tuned models and to set the threshold. The test split is used once, at the end.
5. **Uncertainty on every comparison.** Each AUC and each AUC difference comes with a bootstrap interval that resamples whole lots.
6. **Calibration and the threshold are tied together.** For a calibrated model the cost-minimising threshold should be 1/(1+5) ≈ 0.167. We check whether ours lands there.
7. **Local SHAP next to the wafer map itself**, so each explanation can be checked by eye.

## 1. Setup and data

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from IPython.display import display, Markdown

from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix
from sklearn.base import clone
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")
SEED = 42
C_FN, C_FP = 5.0, 1.0            # evaluation loss: a missed Edge-Loc costs 5 unnecessary tool holds
N_BOOT = 2000                    # bootstrap resamples (whole lots)

DATA = "data/wm811k_edgeloc_loc_features.csv"
MAPS = "data/wm811k_edgeloc_loc_maps.npz"
FIG = "figures"; os.makedirs(FIG, exist_ok=True)

# Plot style: glassbox / Loc = blue, blackbox / Edge-Loc = orange; solid = log loss, dashed = L2
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "axes.axisbelow": True, "legend.fontsize": 8, "figure.dpi": 110,
    "lines.linewidth": 2,
})
def savefig(name):
    plt.savefig(os.path.join(FIG, name), dpi=200, bbox_inches="tight")

df = pd.read_csv(DATA)
FEATURES = ["n_dies", "fail_rate", "center_fail_rate", "middle_fail_rate", "edge_fail_rate", "outer_ring_fail_rate",
            "n_fail_clusters", "largest_cluster_share", "largest_cluster_radius", "angular_concentration"]
TARGET = "edge_loc"
X, y, lots = df[FEATURES], df[TARGET], df["lot"]
print(f"{len(df):,} wafers, {lots.nunique():,} lots, {len(FEATURES)} features, Edge-Loc rate {y.mean():.1%}")
df.head()

8,782 wafers, 5,008 lots, 10 features, Edge-Loc rate 59.1%


,wm811k_row,lot,wafer_index,n_dies,fail_rate,center_fail_rate,middle_fail_rate,edge_fail_rate,outer_ring_fail_rate,n_fail_clusters,largest_cluster_share,largest_cluster_radius,angular_concentration,pattern,edge_loc
0,19,lot1,20,1683,0.086750,0.062112,0.071031,0.105721,0.148305,65,0.037433,0.697603,0.499021,Loc,0
1,36,lot2,12,2460,0.089024,0.057851,0.046139,0.134948,0.310811,124,0.021951,0.941795,0.168409,Edge-Loc,1
2,37,lot2,13,2460,0.097154,0.074380,0.054614,0.141003,0.310811,125,0.024390,0.937394,0.179471,Edge-Loc,1
3,38,lot2,14,2460,0.109756,0.103306,0.050847,0.165225,0.361486,126,0.032927,0.923832,0.263500,Edge-Loc,1
4,39,lot2,15,2460,0.111382,0.028926,0.051789,0.183391,0.314189,110,0.026016,0.931547,0.329178,Edge-Loc,1


**The data.** WM-811K has 811,457 wafer maps from a real fab, of which 172,950 were labelled by engineers. We use the 8,782 wafers labelled **Edge-Loc** (5,189) or **Loc** (3,593). `build_features.py` turns each map into the 10 features below and saves them to a small CSV, so this notebook doesn't need the 2 GB source file.

| Feature | Meaning |
|---|---|
| `n_dies` | Dies on the wafer (a proxy for product and die size) |
| `fail_rate` | Share of dies that fail |
| `center_fail_rate`, `middle_fail_rate`, `edge_fail_rate`, `outer_ring_fail_rate` | Fail rate in the centre (radius < 0.3), middle (0.3–0.7), edge (≥ 0.7) and outermost ring (≥ 0.9); the radius is 1 at the outermost die |
| `n_fail_clusters` | Number of connected groups of failing dies |
| `largest_cluster_share` | Size of the largest cluster, as a share of all dies |
| `largest_cluster_radius` | Mean radius of the largest cluster (0 = centre, 1 = edge) |
| `angular_concentration` | 0 if failures are spread all around the wafer, 1 if they sit on one side |

In [2]:
# Data checks: all of these use features only, no labels
checks = pd.Series({
    "missing cells": int(df[FEATURES + [TARGET]].isna().sum().sum()),
    "duplicate feature rows": int(df.duplicated(subset=FEATURES).sum()),
    "duplicate feature rows with conflicting labels": int((df.groupby(FEATURES)[TARGET].nunique() > 1).sum()),
    "constant features": int((df[FEATURES].nunique() <= 1).sum()),
    "wafers per lot (median)": float(df.groupby("lot").size().median()),
    "wafers per lot (max)": int(df.groupby("lot").size().max()),
})
display(checks.to_frame("value"))
display(df[FEATURES].describe().T[["mean", "std", "min", "50%", "max"]].round(3))

,value
missing cells,0.0
duplicate feature rows,55.0
duplicate feature rows with conflicting labels,0.0
constant features,0.0
wafers per lot (median),1.0
wafers per lot (max),25.0


,mean,std,min,50%,max
n_dies,1244.742,1293.059,464.000,939.000,20761.000
fail_rate,0.169,0.105,0.005,0.150,0.746
center_fail_rate,0.148,0.147,0.000,0.106,1.000
middle_fail_rate,0.129,0.114,0.000,0.099,0.879
edge_fail_rate,0.214,0.120,0.000,0.197,0.940
outer_ring_fail_rate,0.319,0.159,0.000,0.309,1.000
n_fail_clusters,53.713,32.237,1.000,49.000,591.000
largest_cluster_share,0.069,0.092,0.002,0.037,0.702
largest_cluster_radius,0.734,0.187,0.101,0.794,0.984
angular_concentration,0.259,0.155,0.003,0.230,0.999


## 2. Splits: train / validation / test, by lot
Wafers from the same lot went through the same tools together, so they resemble each other. A random split would put some wafers of a lot in training and others in validation, which makes validation look better than it will on new lots. We split **by lot** (60 / 20 / 20), stratified on the label.

- **Train:** fit the models; tune hyperparameters by 5-fold cross-validation (also by lot).
- **Validation:** compare the four tuned models on AUC, choose one, set its threshold.
- **Test:** touched once, in section 10.

In [3]:
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X, y, lots))
test_idx, val_idx = folds[0][1], folds[1][1]
train_idx = np.setdiff1d(np.arange(len(df)), np.r_[test_idx, val_idx])
assert not set(lots.iloc[train_idx]) & set(lots.iloc[val_idx]) and not set(lots.iloc[train_idx]) & set(lots.iloc[test_idx])

X_tr, y_tr, g_tr = X.iloc[train_idx], y.iloc[train_idx], lots.iloc[train_idx]
X_va, y_va, g_va = X.iloc[val_idx], y.iloc[val_idx], lots.iloc[val_idx]
X_te, y_te, g_te = X.iloc[test_idx], y.iloc[test_idx], lots.iloc[test_idx]
display(pd.DataFrame({"wafers": [len(train_idx), len(val_idx), len(test_idx)],
                      "lots": [g_tr.nunique(), g_va.nunique(), g_te.nunique()],
                      "Edge-Loc": [int(y_tr.sum()), int(y_va.sum()), int(y_te.sum())],
                      "Edge-Loc rate": [f"{v:.1%}" for v in (y_tr.mean(), y_va.mean(), y_te.mean())]},
                     index=["train", "validation", "test"]))

,wafers,lots,Edge-Loc,Edge-Loc rate
train,5268,2996,3112,59.1%
validation,1756,1003,1038,59.1%
test,1758,1009,1039,59.1%


### A look at the training data
Only the training split is plotted, so nothing here leaks from validation or test.

In [4]:
fig, axes = plt.subplots(2, 5, figsize=(13, 4.6)); axes = axes.ravel()
for ax, c in zip(axes, FEATURES):
    v = X_tr[c]; hi = v.quantile(0.99) if c in ("n_dies", "n_fail_clusters") else v.max()
    bins = np.linspace(v.min(), hi, 31)
    ax.hist(v[y_tr == 0], bins=bins, density=True, histtype="step", lw=1.8, color=BLUE, label="Loc")
    ax.hist(v[y_tr == 1], bins=bins, density=True, histtype="step", lw=1.8, color=ORANGE, label="Edge-Loc")
    ax.set_title(c, fontsize=9); ax.set_yticks([])
axes[0].legend()
fig.suptitle("Feature distributions by class (training split)", fontweight="bold"); plt.tight_layout(); savefig("01_features_by_class.png"); plt.show()

corr = X_tr.corr()
hi_pairs = [(a, b, round(corr.loc[a, b], 2)) for i, a in enumerate(FEATURES) for b in FEATURES[i + 1:] if abs(corr.loc[a, b]) >= 0.85]
print("feature pairs with |r| ≥ 0.85 (training split):", hi_pairs)

feature pairs with |r| ≥ 0.85 (training split): [('fail_rate', 'middle_fail_rate', np.float64(0.91)), ('fail_rate', 'edge_fail_rate', np.float64(0.88)), ('fail_rate', 'largest_cluster_share', np.float64(0.87)), ('middle_fail_rate', 'largest_cluster_share', np.float64(0.85)), ('edge_fail_rate', 'outer_ring_fail_rate', np.float64(0.85))]


The zone fail rates are strongly correlated with the overall fail rate. That matters twice later: an L1 penalty is the right choice for the linear glassbox (it drops redundant features, which is why the slides prefer LASSO to ridge), and SHAP credit can be shared arbitrarily between correlated features (section 9).

## 3. The evaluation loss
A function of the decision threshold: flag a wafer as Edge-Loc when its score is at least `t`.

In [5]:
def eval_loss(y_true, score, t, c_fn=C_FN, c_fp=C_FP):
    """Cost per wafer when wafers with score >= t are flagged as Edge-Loc."""
    y_true = np.asarray(y_true); flag = np.asarray(score) >= t
    fn = np.sum((y_true == 1) & ~flag); fp = np.sum((y_true == 0) & flag)
    return (c_fn * fn + c_fp * fp) / len(y_true)

def best_threshold(y_true, score, c_fn=C_FN, c_fp=C_FP):
    """Exact minimiser over all thresholds that change a decision. Returns (threshold, cost)."""
    y_true = np.asarray(y_true); score = np.asarray(score)
    order = np.argsort(-score); s, yy = score[order], y_true[order]
    tp = np.cumsum(yy); fp = np.cumsum(1 - yy)                # flag the k highest scores
    cost = np.r_[c_fn * yy.sum(), c_fn * (yy.sum() - tp) + c_fp * fp] / len(yy)
    last = np.r_[True, np.r_[s[1:] != s[:-1], True]]          # only cut between distinct scores
    k = np.flatnonzero(last)[np.argmin(cost[last])]
    t = np.inf if k == 0 else (s[k - 1] if k == len(s) else (s[k - 1] + s[k]) / 2)
    return float(t), float(cost[k])

T_THEORY = C_FP / (C_FP + C_FN)
base = y_va.mean()
print(f"If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = {T_THEORY:.3f}.")
print(f"Reference policies on validation: hold the tool for every wafer = {C_FP * (1 - base):.3f} per wafer; "
      f"never hold = {C_FN * base:.3f} per wafer.")

If the model's probabilities are calibrated, the cost-minimising threshold is C_FP / (C_FP + C_FN) = 0.167.
Reference policies on validation: hold the tool for every wafer = 0.409 per wafer; never hold = 2.956 per wafer.


## 4. Four models: glassbox and blackbox, each with L2 loss and with log loss
| | L2 loss (squared error on the 0/1 label) | Surrogate: binomial deviance (log loss) |
|---|---|---|
| **Glassbox:** L1-penalised linear model on standardised features | LASSO regression | L1 logistic regression |
| **Blackbox:** gradient-boosted trees | boosting with squared error | boosting with log loss |

Inside each row, the model structure and the search grid are the same and **only the training loss changes**, so the comparison isolates the loss.

**Tuning.** Each of the four is tuned separately by 5-fold cross-validated AUC inside the training split (folds by lot). The grids are deliberately small: one knob for the glassbox (penalty strength, 6 values) and two for the blackbox (tree depth × number of trees, 6 combinations). That's 24 configurations in total. The validation split plays no part in tuning, so it's only asked one question: which of four tuned models is best?

In [6]:
def score_of(model, X_):
    """One score per wafer: P(Edge-Loc) for log-loss models, the fitted value for L2 models."""
    return model.predict_proba(X_)[:, 1] if hasattr(model, "predict_proba") else model.predict(X_)

def auc_scorer(model, X_, y_):
    return roc_auc_score(y_, score_of(model, X_))

SPECS = {
    ("glassbox", "L2"): (make_pipeline(StandardScaler(), Lasso(max_iter=10000)),
                         {"lasso__alpha": [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1]}),
    ("glassbox", "log loss"): (make_pipeline(StandardScaler(), LogisticRegression(penalty="l1", solver="liblinear", max_iter=2000, random_state=SEED)),
                               {"logisticregression__C": [0.003, 0.01, 0.03, 0.1, 1.0, 10.0]}),
    ("blackbox", "L2"): (HistGradientBoostingRegressor(loss="squared_error", learning_rate=0.1, random_state=SEED),
                         {"max_depth": [2, 3, 5], "max_iter": [100, 300]}),
    ("blackbox", "log loss"): (HistGradientBoostingClassifier(learning_rate=0.1, random_state=SEED),
                               {"max_depth": [2, 3, 5], "max_iter": [100, 300]}),
}
inner_cv = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y_tr, g_tr))

MODELS, SEARCH, rows = {}, {}, []
for key, (est, grid) in SPECS.items():
    gs = GridSearchCV(est, grid, scoring=auc_scorer, cv=inner_cv, n_jobs=-1, refit=True).fit(X_tr, y_tr)
    MODELS[key] = gs.best_estimator_; SEARCH[key] = gs
    rows.append({"model": key[0], "train loss": key[1], "configurations tried": len(gs.cv_results_["params"]),
                 "chosen": ", ".join(f"{k.split('__')[-1]}={v}" for k, v in gs.best_params_.items()),
                 "CV AUC (train split)": round(gs.best_score_, 4)})
tuning = pd.DataFrame(rows); display(tuning)
print("total configurations tried:", int(tuning["configurations tried"].sum()))

,model,train loss,configurations tried,chosen,CV AUC (train split)
0,glassbox,L2,6,alpha=0.0003,0.9595
1,glassbox,log loss,6,C=10.0,0.9686
2,blackbox,L2,6,"max_depth=3, max_iter=300",0.9686
3,blackbox,log loss,6,"max_depth=5, max_iter=300",0.9716


total configurations tried: 24


### How much does the glassbox penalty matter?
The L1 penalty is what keeps a linear model readable: a stronger penalty leaves fewer features. This table shows every penalty value tried for the L1 logistic model, with its cross-validated AUC on the training split and the number of features it keeps. It uses the training split only.

In [7]:
gs = SEARCH[("glassbox", "log loss")]
rows = []
for params, mean, sd in zip(gs.cv_results_["params"], gs.cv_results_["mean_test_score"], gs.cv_results_["std_test_score"]):
    m = clone(SPECS[("glassbox", "log loss")][0]).set_params(**params).fit(X_tr, y_tr)
    kept = [f for f, c in zip(FEATURES, m[-1].coef_.ravel()) if c != 0]
    rows.append({"C (larger = weaker penalty)": params["logisticregression__C"], "CV AUC": round(mean, 4), "sd over folds": round(sd, 4),
                 "features kept": len(kept), "dropped": ", ".join(f for f in FEATURES if f not in kept) or "none"})
display(pd.DataFrame(rows))

,C (larger = weaker penalty),CV AUC,sd over folds,features kept,dropped
0,0.003,0.9480,0.0081,3,"n_dies, fail_rate, center_fail_rate, middle_fa..."
1,0.010,0.9537,0.0067,4,"n_dies, fail_rate, center_fail_rate, middle_fa..."
2,0.030,0.9573,0.0063,8,"fail_rate, center_fail_rate"
3,0.100,0.9602,0.0063,9,center_fail_rate
4,1.000,0.9680,0.0033,10,none
5,10.000,0.9686,0.0031,10,none


## 5. Compare on validation AUC and decide
AUC is computed on the validation split, which no model has seen. The intervals come from a bootstrap that resamples **whole lots** 2,000 times, and differences between models are computed on the same resamples (paired).

**Decision rule, fixed before looking at the results:**
1. *Within a family (L2 vs log loss):* take the higher validation AUC. If the difference isn't distinguishable from zero, take the log-loss version, because its output is a probability.
2. *Between families:* a blackbox has to earn its opacity. Choose it only if its validation AUC is distinguishably higher **and** that lowers the evaluation loss by at least 10%. Otherwise choose the glassbox.

In [8]:
S_va = {k: score_of(m, X_va) for k, m in MODELS.items()}
S_te = {k: score_of(m, X_te) for k, m in MODELS.items()}    # computed now, not looked at until section 10

def lot_bootstrap(groups, n_boot=N_BOOT, seed=SEED):
    """Row indices for n_boot resamples of whole lots."""
    groups = np.asarray(groups); rng = np.random.default_rng(seed)
    by_lot = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    for _ in range(n_boot):
        yield np.concatenate([by_lot[i] for i in rng.integers(0, len(by_lot), len(by_lot))])

def boot_stats(y_true, groups, fns):
    """Bootstrap each statistic in fns (name -> function of row indices) on the same resamples."""
    y_true = np.asarray(y_true); out = {k: [] for k in fns}
    for idx in lot_bootstrap(groups):
        if y_true[idx].min() == y_true[idx].max(): continue
        for k, f in fns.items(): out[k].append(f(idx))
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in out.items()}

yv = np.asarray(y_va)
ci = boot_stats(yv, g_va, {k: (lambda idx, s=S_va[k]: roc_auc_score(yv[idx], s[idx])) for k in MODELS})
auc_tab = pd.DataFrame([{"model": k[0], "train loss": k[1], "validation AUC": roc_auc_score(yv, S_va[k]),
                         "95% CI low": ci[k][0], "95% CI high": ci[k][1]} for k in MODELS])
display(auc_tab.round(4))

def auc_diff(a, b):
    d = roc_auc_score(yv, S_va[a]) - roc_auc_score(yv, S_va[b])
    lo, hi = boot_stats(yv, g_va, {"d": lambda idx: roc_auc_score(yv[idx], S_va[a][idx]) - roc_auc_score(yv[idx], S_va[b][idx])})["d"]
    return d, lo, hi

# Rule 1: within each family
best = {}
for fam in ("glassbox", "blackbox"):
    d, lo, hi = auc_diff((fam, "log loss"), (fam, "L2"))
    best[fam] = (fam, "L2") if hi < 0 else (fam, "log loss")
    print(f"{fam}: log loss − L2 validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f}) -> keep {best[fam][1]}")

# Rule 2: between families
d, lo, hi = auc_diff(best["blackbox"], best["glassbox"])
cost_g = best_threshold(yv, S_va[best["glassbox"]])[1]; cost_b = best_threshold(yv, S_va[best["blackbox"]])[1]
saving = 1 - cost_b / cost_g
print(f"\nblackbox − glassbox validation AUC = {d:+.4f} (95% CI {lo:+.4f} to {hi:+.4f})")
print(f"evaluation loss at each model's best threshold: glassbox {cost_g:.4f}, blackbox {cost_b:.4f} per wafer "
      f"-> the blackbox lowers it by {saving:.1%}")
CHOSEN = best["blackbox"] if (lo > 0 and saving >= 0.10) else best["glassbox"]
OTHER = best["glassbox"] if CHOSEN[0] == "blackbox" else best["blackbox"]
display(Markdown(f"**Decision: {CHOSEN[0]}, trained with {CHOSEN[1]}.**"))

,model,train loss,validation AUC,95% CI low,95% CI high
0,glassbox,L2,0.9581,0.9474,0.9675
1,glassbox,log loss,0.9659,0.9565,0.9741
2,blackbox,L2,0.9655,0.9538,0.9748
3,blackbox,log loss,0.9720,0.9634,0.9796


glassbox: log loss − L2 validation AUC = +0.0078 (95% CI +0.0027 to +0.0135) -> keep log loss


blackbox: log loss − L2 validation AUC = +0.0065 (95% CI +0.0025 to +0.0114) -> keep log loss



blackbox − glassbox validation AUC = +0.0061 (95% CI +0.0001 to +0.0122)
evaluation loss at each model's best threshold: glassbox 0.1429, blackbox 0.1492 per wafer -> the blackbox lowers it by -4.4%


**Decision: glassbox, trained with log loss.**

In [9]:
STYLE = {("glassbox", "L2"): (BLUE, "--"), ("glassbox", "log loss"): (BLUE, "-"),
         ("blackbox", "L2"): (ORANGE, "--"), ("blackbox", "log loss"): (ORANGE, "-")}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for k, s in S_va.items():
    fpr, tpr, _ = roc_curve(yv, s)
    for ax in axes: ax.plot(fpr, tpr, color=STYLE[k][0], ls=STYLE[k][1], lw=1.8, label=f"{k[0]}, {k[1]} (AUC {roc_auc_score(yv, s):.3f})")
axes[0].plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
axes[0].set_title("ROC on validation"); axes[0].set_xlabel("false positive rate"); axes[0].set_ylabel("true positive rate"); axes[0].legend(loc="lower right")
axes[1].set_xlim(0, 0.3); axes[1].set_ylim(0.7, 1.0); axes[1].set_title("Zoom: the top-left corner"); axes[1].set_xlabel("false positive rate")
plt.tight_layout(); savefig("02_roc_validation.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 2.4))
for i, r in auc_tab.iloc[::-1].reset_index(drop=True).iterrows():
    k = (r["model"], r["train loss"])
    ax.plot([r["95% CI low"], r["95% CI high"]], [i, i], color=STYLE[k][0], lw=2, ls=STYLE[k][1])
    ax.plot(r["validation AUC"], i, "o", ms=8, color=STYLE[k][0], mec="#fcfcfb", mew=2)
    ax.text(r["95% CI high"] + 0.0008, i, f"{r['validation AUC']:.3f}", va="center", fontsize=8, color=INK2)
ax.set_yticks(range(4)); ax.set_yticklabels([f"{r['model']}, {r['train loss']}" for _, r in auc_tab.iloc[::-1].iterrows()])
ax.set_xlabel("validation AUC with 95% lot-bootstrap interval"); ax.grid(axis="y", visible=False); ax.set_title("Four tuned models on validation")
plt.tight_layout(); savefig("03_auc_intervals.png"); plt.show()

## 6. Threshold for the chosen model
For the chosen model we take the threshold that minimises the evaluation loss on validation, and compare it with the default 0.5.

In [10]:
s_va = S_va[CHOSEN]
T_STAR, cost_star = best_threshold(yv, s_va)
cost_05 = eval_loss(yv, s_va, 0.5)
ts = np.linspace(0.005, 0.995, 199)
curve = np.array([eval_loss(yv, s_va, t) for t in ts])
curve_mis = np.array([eval_loss(yv, s_va, t, 1, 1) for t in ts])
t_mis = best_threshold(yv, s_va, 1, 1)[0]

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(ts, curve, color=ORANGE, label=f"evaluation loss (FN = {C_FN:g}, FP = {C_FP:g})")
ax.plot(ts, curve_mis, color=BLUE, label="misclassification rate (FN = FP = 1)")
ax.axvline(T_STAR, color=INK2, lw=1, ls="--"); ax.text(T_STAR, curve.max() * 0.97, f" best: {T_STAR:.3f}", fontsize=8, color=INK2, va="top")
ax.axvline(0.5, color=MUTED, lw=1, ls=":"); ax.text(0.5, curve.max() * 0.97, " default 0.5", fontsize=8, color=MUTED, va="top")
ax.set_xlabel(f"threshold on the {CHOSEN[0]} score"); ax.set_ylabel("cost per wafer"); ax.set_title("Evaluation loss vs threshold (validation)"); ax.legend(loc="upper center")
plt.tight_layout(); savefig("04_threshold.png"); plt.show()

flag = s_va >= T_STAR
tn, fp, fn, tp = confusion_matrix(yv, flag).ravel()
print(f"best threshold = {T_STAR:.3f}: cost {cost_star:.4f} per wafer (FN {fn}, FP {fp}; recall {tp / (tp + fn):.1%}, tool holds {flag.mean():.1%} of wafers)")
tn5, fp5, fn5, tp5 = confusion_matrix(yv, s_va >= 0.5).ravel()
print(f"threshold 0.5:           cost {cost_05:.4f} per wafer (FN {fn5}, FP {fp5}; recall {tp5 / (tp5 + fn5):.1%})")
print(f"-> tuning the threshold lowers the evaluation loss by {1 - cost_star / cost_05:.1%}. "
      f"The threshold that minimises plain misclassification would be {t_mis:.3f}.")
t_lo, t_hi = np.percentile([best_threshold(yv[i], s_va[i])[0] for i in lot_bootstrap(g_va, 500)], [2.5, 97.5])
print(f"bootstrap 95% range of the best threshold: {t_lo:.3f} to {t_hi:.3f} (theory for a calibrated model: {T_THEORY:.3f})")

best threshold = 0.279: cost 0.1429 per wafer (FN 18, FP 161; recall 98.3%, tool holds 67.3% of wafers)
threshold 0.5:           cost 0.1959 per wafer (FN 49, FP 99; recall 95.3%)
-> tuning the threshold lowers the evaluation loss by 27.0%. The threshold that minimises plain misclassification would be 0.614.
bootstrap 95% range of the best threshold: 0.235 to 0.396 (theory for a calibrated model: 0.167)


## 7. Calibration
A model is calibrated if, among wafers given a score near 0.3, about 30% are Edge-Loc. We measure it on validation with a reliability diagram, the expected calibration error (ECE: the average gap between predicted and observed rates, over 10 bins) and the Brier score.

**The fix.** We fit a calibrator on **out-of-fold training scores** (5 folds by lot, same hyperparameters), then apply it to validation. Fitting the calibrator on validation would reuse the data that also sets the threshold. Two calibrators are tried: Platt scaling (a one-feature logistic regression, two parameters) and isotonic regression (a monotone step function).

In [11]:
def ece(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    b = np.clip((p * bins).astype(int), 0, bins - 1)
    return float(sum((b == i).mean() * abs(y_true[b == i].mean() - p[b == i].mean()) for i in range(bins) if (b == i).any()))

def reliability(y_true, p, bins=10):
    y_true = np.asarray(y_true); p = np.clip(np.asarray(p, float), 0, 1)
    q = np.unique(np.quantile(p, np.linspace(0, 1, bins + 1)))               # equal-count bins
    b = np.clip(np.digitize(p, q[1:-1]), 0, len(q) - 2)
    return (np.array([p[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]),
            np.array([y_true[b == i].mean() for i in range(len(q) - 1) if (b == i).any()]))

# Out-of-fold training scores for every model, with its tuned hyperparameters
OOF = {}
for k, m in MODELS.items():
    method = "predict_proba" if hasattr(m, "predict_proba") else "predict"
    o = cross_val_predict(clone(m), X_tr, y_tr, cv=inner_cv, method=method, n_jobs=-1)
    OOF[k] = o[:, 1] if o.ndim == 2 else o

def to_feature(k, s):
    """Platt input: the logit for probability models, the raw score for L2 models."""
    s = np.asarray(s, float)
    if k[1] == "L2": return s.reshape(-1, 1)
    s = np.clip(s, 1e-6, 1 - 1e-6); return np.log(s / (1 - s)).reshape(-1, 1)

CAL = {}
for k in MODELS:
    platt = LogisticRegression(C=1e6, max_iter=1000).fit(to_feature(k, OOF[k]), y_tr)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(OOF[k], y_tr)
    CAL[k] = {"Platt": (lambda s, k=k, m=platt: m.predict_proba(to_feature(k, s))[:, 1]),
              "isotonic": (lambda s, m=iso: m.predict(np.asarray(s, float)))}

rows = []
for k in MODELS:
    raw = S_va[k]
    row = {"model": k[0], "train loss": k[1], "scores outside [0, 1]": f"{np.mean((raw < 0) | (raw > 1)):.1%}",
           "ECE raw": ece(yv, raw), "ECE Platt": ece(yv, CAL[k]["Platt"](raw)), "ECE isotonic": ece(yv, CAL[k]["isotonic"](raw)),
           "Brier raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Brier Platt": brier_score_loss(yv, CAL[k]["Platt"](raw)),
           "Brier isotonic": brier_score_loss(yv, CAL[k]["isotonic"](raw))}
    rows.append(row)
cal_tab = pd.DataFrame(rows); display(cal_tab.round(4))

def ece_ci(p):
    return np.percentile([ece(yv[i], p[i]) for i in lot_bootstrap(g_va, 500)], [2.5, 97.5])
raw = S_va[CHOSEN]
for name, p in [("raw", raw), ("Platt", CAL[CHOSEN]["Platt"](raw)), ("isotonic", CAL[CHOSEN]["isotonic"](raw))]:
    lo, hi = ece_ci(p); print(f"chosen model, {name:8s}: ECE {ece(yv, p):.4f} (95% CI {lo:.4f} to {hi:.4f})")

,model,train loss,"scores outside [0, 1]",ECE raw,ECE Platt,ECE isotonic,Brier raw,Brier Platt,Brier isotonic
0,glassbox,L2,19.9%,0.1067,0.0200,0.0175,0.0939,0.0766,0.0769
1,glassbox,log loss,0.0%,0.0309,0.0286,0.0212,0.0654,0.0655,0.0645
2,blackbox,L2,29.4%,0.0261,0.0193,0.0243,0.0673,0.0657,0.0661
3,blackbox,log loss,0.0%,0.0339,0.0133,0.0139,0.0634,0.0618,0.0620


chosen model, raw     : ECE 0.0309 (95% CI 0.0236 to 0.0458)


chosen model, Platt   : ECE 0.0286 (95% CI 0.0211 to 0.0443)


chosen model, isotonic: ECE 0.0212 (95% CI 0.0154 to 0.0358)


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
for k in MODELS:
    mp, ob = reliability(yv, S_va[k]); ax.plot(mp, ob, marker="o", ms=4, lw=1.6, color=STYLE[k][0], ls=STYLE[k][1], label=f"{k[0]}, {k[1]}")
ax.set_title("Before calibration: all four models"); ax.set_xlabel("mean predicted probability (clipped to 0–1)"); ax.set_ylabel("observed Edge-Loc rate"); ax.legend()
ax = axes[1]; ax.plot([0, 1], [0, 1], color=MUTED, lw=1, ls=":")
col = STYLE[CHOSEN][0]
for name, p, ls, mk in [("raw", raw, "-", "o"), ("Platt", CAL[CHOSEN]["Platt"](raw), "--", "s"), ("isotonic", CAL[CHOSEN]["isotonic"](raw), ":", "^")]:
    mp, ob = reliability(yv, p); ax.plot(mp, ob, marker=mk, ms=5, lw=1.6, color=col if name == "raw" else INK2, ls=ls, label=f"{name} (ECE {ece(yv, p):.3f})")
ax.set_title(f"Chosen model ({CHOSEN[0]}, {CHOSEN[1]}): before and after"); ax.set_xlabel("mean predicted probability"); ax.legend()
plt.tight_layout(); savefig("05_calibration.png"); plt.show()

# Pick the calibrator by validation Brier score (a proper scoring rule); keep raw if neither helps
briers = {"raw": brier_score_loss(yv, np.clip(raw, 0, 1)), "Platt": brier_score_loss(yv, CAL[CHOSEN]["Platt"](raw)),
          "isotonic": brier_score_loss(yv, CAL[CHOSEN]["isotonic"](raw))}
CALIB = min(briers, key=briers.get)
calibrate = (lambda s: np.clip(np.asarray(s, float), 0, 1)) if CALIB == "raw" else CAL[CHOSEN][CALIB]
p_va = calibrate(raw)
T_CAL, cost_cal = best_threshold(yv, p_va)
print(f"calibrator with the lowest validation Brier score: {CALIB}  {({k: round(v, 4) for k, v in briers.items()})}")
print(f"AUC before {roc_auc_score(yv, raw):.4f}, after {roc_auc_score(yv, p_va):.4f} (a monotone map can only change AUC through ties)")
print(f"best threshold on calibrated probabilities = {T_CAL:.3f} (cost {cost_cal:.4f}); theory says {T_THEORY:.3f}, "
      f"which costs {eval_loss(yv, p_va, T_THEORY):.4f} on validation")

calibrator with the lowest validation Brier score: isotonic  {'raw': 0.0654, 'Platt': 0.0655, 'isotonic': 0.0645}
AUC before 0.9659, after 0.9659 (a monotone map can only change AUC through ties)
best threshold on calibrated probabilities = 0.220 (cost 0.1429); theory says 0.167, which costs 0.1475 on validation


**Why this matters for the threshold.** If the probabilities are calibrated, we don't need to search for a threshold at all: decision theory gives it as C_FP / (C_FP + C_FN) = 0.167, and it updates by formula when the costs change. The last line above checks how close the searched threshold on calibrated probabilities is to that value, and what using the formula would cost.

**The final decision rule** is: calibrate the chosen model's score, then hold the tool when the calibrated probability is at least the threshold found on validation.

## 8. How sensitive is the decision to the cost ratio?
The 5:1 ratio is our assumption. Here the ratio runs from 1 (plain misclassification) to 10. For each ratio we show the best threshold and the resulting cost, for the chosen model and for the best model of the other family.

In [13]:
ratios = np.arange(1, 11)
p_other = CAL[OTHER][CALIB](S_va[OTHER]) if CALIB != "raw" else np.clip(S_va[OTHER], 0, 1)
sens = pd.DataFrame({"FN:FP ratio": ratios,
                     "best threshold (chosen)": [best_threshold(yv, p_va, r, 1)[0] for r in ratios],
                     "theory 1/(1+ratio)": 1 / (1 + ratios),
                     f"cost, chosen ({CHOSEN[0]})": [best_threshold(yv, p_va, r, 1)[1] for r in ratios],
                     f"cost, other ({OTHER[0]})": [best_threshold(yv, p_other, r, 1)[1] for r in ratios],
                     "cost, hold every wafer": [(1 - yv.mean())] * len(ratios)})
display(sens.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
axes[0].plot(ratios, sens["best threshold (chosen)"], "o-", color=STYLE[CHOSEN][0], label="best threshold on validation")
axes[0].plot(ratios, sens["theory 1/(1+ratio)"], color=MUTED, ls=":", label="theory for a calibrated model")
axes[0].set_xlabel("cost of a missed Edge-Loc, in unnecessary tool holds"); axes[0].set_ylabel("threshold"); axes[0].set_title("Threshold vs cost ratio"); axes[0].legend()
axes[1].plot(ratios, sens[f"cost, chosen ({CHOSEN[0]})"], "o-", color=STYLE[CHOSEN][0], label=f"chosen: {CHOSEN[0]}")
axes[1].plot(ratios, sens[f"cost, other ({OTHER[0]})"], "s-", color=STYLE[OTHER][0], label=f"other: {OTHER[0]}")
axes[1].axvline(C_FN / C_FP, color=INK2, lw=1, ls="--"); axes[1].text(C_FN / C_FP, axes[1].get_ylim()[0], " our assumption", fontsize=8, color=INK2, va="bottom")
axes[1].set_xlabel("cost of a missed Edge-Loc, in unnecessary tool holds"); axes[1].set_ylabel("cost per wafer at the best threshold"); axes[1].set_title("Evaluation loss vs cost ratio"); axes[1].legend()
plt.tight_layout(); savefig("06_cost_sensitivity.png"); plt.show()

,FN:FP ratio,best threshold (chosen),theory 1/(1+ratio),"cost, chosen (glassbox)","cost, other (blackbox)","cost, hold every wafer"
0,1,0.584,0.500,0.081,0.080,0.409
1,2,0.258,0.333,0.105,0.102,0.409
2,3,0.258,0.250,0.121,0.124,0.409
3,4,0.220,0.200,0.133,0.142,0.409
4,5,0.220,0.167,0.143,0.155,0.409
5,6,0.220,0.143,0.153,0.166,0.409
6,7,0.220,0.125,0.163,0.174,0.409
7,8,0.220,0.111,0.174,0.178,0.409
8,9,0.220,0.100,0.184,0.183,0.409
9,10,0.220,0.091,0.194,0.187,0.409


## 9. Interpretability: tree SHAP on the tree-based blackbox, global and local
Whichever model we chose, the brief asks for tree SHAP on a tree-based blackbox. We explain the **gradient-boosted trees trained with log loss**. SHAP values are in log-odds: a value of +1 for a feature means that feature moved this wafer's log-odds of Edge-Loc up by 1 from the average prediction.

We also put the glassbox's own explanation (its coefficients) next to the global SHAP ranking. If two very different models lean on the same features, that's evidence the signal is in the data, not an artefact of one model.

In [14]:
import shap
gbm = MODELS[("blackbox", "log loss")]
explainer = shap.TreeExplainer(gbm)
sv = np.asarray(explainer.shap_values(X_va))
if sv.ndim == 3: sv = sv[..., 1]
base_value = float(np.ravel(explainer.expected_value)[-1])
p_gbm = S_va[("blackbox", "log loss")]
logit = np.log(np.clip(p_gbm, 1e-9, 1 - 1e-9) / (1 - np.clip(p_gbm, 1e-9, 1 - 1e-9)))
print(f"additivity check: max |base + sum(SHAP) − model log-odds| = {np.abs(base_value + sv.sum(1) - logit).max():.2e}")

glob = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values()
lin = MODELS[("glassbox", "log loss")]
coef = pd.Series(lin[-1].coef_.ravel(), index=FEATURES)
rho = spearmanr(glob[FEATURES], coef.abs()[FEATURES]).correlation

seq = LinearSegmentedColormap.from_list("seq", ["#cde2fb", "#2a78d6", "#0d366b"])
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1, 1.5, 1]})
axes[0].barh(glob.index, glob.values, color=ORANGE, height=0.6); axes[0].grid(axis="y", visible=False)
axes[0].set_title("Global: mean |SHAP| (blackbox)"); axes[0].set_xlabel("mean |SHAP| (log-odds)")
rng = np.random.default_rng(SEED)
for i, f in enumerate(glob.index):
    j = FEATURES.index(f); v = X_va[f].rank(pct=True).values
    sc = axes[1].scatter(sv[:, j], i + rng.uniform(-0.3, 0.3, len(sv)), c=v, cmap=seq, s=5, alpha=0.7, linewidths=0)
axes[1].axvline(0, color=MUTED, lw=1); axes[1].set_yticks(range(len(glob))); axes[1].set_yticklabels(glob.index); axes[1].grid(axis="y", visible=False)
axes[1].set_title("Global: every validation wafer"); axes[1].set_xlabel("SHAP value (log-odds; right = towards Edge-Loc)")
cb = fig.colorbar(sc, ax=axes[1], pad=0.01, shrink=0.8); cb.set_label("feature value (percentile)", fontsize=8); cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"])
c2 = coef[glob.index]
axes[2].barh(c2.index, c2.values, color=BLUE, height=0.6); axes[2].axvline(0, color=MUTED, lw=1); axes[2].grid(axis="y", visible=False)
axes[2].set_yticklabels([]); axes[2].set_title("Glassbox: L1 logistic coefficients"); axes[2].set_xlabel("coefficient per standard deviation (log-odds)")
plt.tight_layout(); savefig("07_shap_global.png"); plt.show()
print("global ranking (mean |SHAP|):", ", ".join(f"{f} {v:.2f}" for f, v in glob.iloc[::-1].items()))
print(f"features the L1 glassbox dropped: {list(coef.index[coef == 0]) or 'none'}")
print(f"rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): {rho:.2f}")

additivity check: max |base + sum(SHAP) − model log-odds| = 4.36e-11


global ranking (mean |SHAP|): largest_cluster_radius 3.98, edge_fail_rate 1.75, outer_ring_fail_rate 0.87, largest_cluster_share 0.82, fail_rate 0.50, n_dies 0.46, middle_fail_rate 0.33, angular_concentration 0.26, center_fail_rate 0.24, n_fail_clusters 0.23
features the L1 glassbox dropped: none
rank agreement between mean |SHAP| and |glassbox coefficient| (Spearman): 0.43


In [15]:
top = glob.index[-1]; second = glob.index[-2]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, f in zip(axes, (top, second)):
    j = FEATURES.index(f)
    ax.scatter(X_va[f][yv == 0], sv[yv == 0, j], s=6, color=BLUE, alpha=0.5, linewidths=0, label="Loc")
    ax.scatter(X_va[f][yv == 1], sv[yv == 1, j], s=6, color=ORANGE, alpha=0.5, linewidths=0, label="Edge-Loc")
    ax.axhline(0, color=MUTED, lw=1); ax.set_xlabel(f); ax.set_ylabel("SHAP value (log-odds)"); ax.set_title(f"How {f} moves the prediction")
    if f in ("n_dies", "n_fail_clusters"): ax.set_xlim(X_va[f].min(), X_va[f].quantile(0.99))
axes[0].legend(markerscale=2.5)
plt.tight_layout(); savefig("08_shap_dependence.png"); plt.show()

### Local explanations, next to the wafer itself
Three validation wafers, chosen by rule (not by hand):
1. the Edge-Loc wafer the blackbox is **most sure** about
2. the Loc wafer it is most sure about
3. its **costliest mistake**: the true Edge-Loc wafer with the lowest predicted probability (a false negative, which costs 5)

Each row shows the wafer map and the SHAP values that add up to that wafer's prediction.

In [16]:
z = np.load(MAPS); map_row = {r: i for i, r in enumerate(z["wm811k_row"])}
wafer_cmap = ListedColormap(["#fcfcfb", "#cde2fb", ORANGE])        # no die, passing die, failing die

def wafer_map(row_id):
    m = z["maps"][map_row[row_id]]
    keep_r = np.flatnonzero((m != 255).any(1)); keep_c = np.flatnonzero((m != 255).any(0))
    return m[: keep_r.max() + 1, : keep_c.max() + 1]

pos, neg = np.flatnonzero(yv == 1), np.flatnonzero(yv == 0)
picks = [("Most confident Edge-Loc", pos[np.argmax(p_gbm[pos])]),
         ("Most confident Loc", neg[np.argmin(p_gbm[neg])]),
         ("Costliest mistake: an Edge-Loc scored as Loc", pos[np.argmin(p_gbm[pos])])]

fig, axes = plt.subplots(len(picks), 2, figsize=(11, 3.3 * len(picks)), gridspec_kw={"width_ratios": [1, 2.6]})
for (title, i), (axm, axs) in zip(picks, axes):
    row = df.iloc[val_idx[i]]
    axm.imshow(wafer_map(row["wm811k_row"]), cmap=wafer_cmap, vmin=0, vmax=2, interpolation="nearest"); axm.axis("off")
    axm.set_title(f"{title}\nlabel: {row['pattern']}, lot {row['lot']}", fontsize=9)
    contrib = pd.Series(sv[i], index=FEATURES); contrib = contrib.reindex(contrib.abs().sort_values().index)
    labels = [f"{f} = {X_va.iloc[i][f]:.3g}" for f in contrib.index]
    axs.barh(labels, contrib.values, color=[ORANGE if v > 0 else BLUE for v in contrib.values], height=0.6)
    axs.axvline(0, color=MUTED, lw=1); axs.grid(axis="y", visible=False)
    axs.set_title(f"base {base_value:+.2f} + SHAP {sv[i].sum():+.2f} = log-odds {logit[i]:+.2f}  →  P(Edge-Loc) = {p_gbm[i]:.3f}", fontsize=9)
    axs.set_xlabel("SHAP value (log-odds; orange pushes towards Edge-Loc, blue towards Loc)")
plt.tight_layout(); savefig("09_shap_local.png"); plt.show()
p_glass = S_va[("glassbox", "log loss")]
for title, i in picks:
    print(f"{title}: blackbox P = {p_gbm[i]:.3f}, glassbox P = {p_glass[i]:.3f}")

Most confident Edge-Loc: blackbox P = 1.000, glassbox P = 0.998
Most confident Loc: blackbox P = 0.000, glassbox P = 0.000
Costliest mistake: an Edge-Loc scored as Loc: blackbox P = 0.000, glassbox P = 0.000


**How far to trust these plots.** The slides warn that every explanation rests on strong assumptions. Three apply here:
- **Correlated features.** The zone fail rates move together (section 2). SHAP splits credit among correlated features in a way that depends on how the trees happened to split, so the ranking *among the fail-rate features* isn't reliable. Their combined story is.
- **SHAP explains the model, not the fab.** A large SHAP value says the model relies on a feature. It doesn't say that changing the feature would change the defect.
- **The labels are human judgements.** Engineers drew the line between "Loc" and "Edge-Loc" by eye. Some of the model's "mistakes" near the boundary may be labelling disagreements, and the third wafer above is worth looking at in that light.

## 10. Final check on the test split (used once)
Everything is frozen: the chosen model, its calibrator and its threshold. The test split has not been used for any decision above.

In [17]:
yt = np.asarray(y_te)
p_te = calibrate(S_te[CHOSEN])
auc_te = roc_auc_score(yt, p_te)
lo, hi = boot_stats(yt, g_te, {"a": lambda idx: roc_auc_score(yt[idx], p_te[idx])})["a"]
flag = p_te >= T_CAL
tn, fp, fn, tp = confusion_matrix(yt, flag).ravel()
cost_te = eval_loss(yt, p_te, T_CAL)
c_lo, c_hi = np.percentile([eval_loss(yt[i], p_te[i], T_CAL) for i in lot_bootstrap(g_te)], [2.5, 97.5])
final = pd.DataFrame({
    "validation": [roc_auc_score(yv, p_va), cost_cal, eval_loss(yv, p_va, 0.5), ece(yv, p_va), C_FP * (1 - yv.mean()), C_FN * yv.mean()],
    "test": [auc_te, cost_te, eval_loss(yt, p_te, 0.5), ece(yt, p_te), C_FP * (1 - yt.mean()), C_FN * yt.mean()],
}, index=["AUC", f"evaluation loss at threshold {T_CAL:.3f}", "evaluation loss at threshold 0.5", "ECE (calibrated)",
          "evaluation loss: hold every wafer", "evaluation loss: never hold"])
display(final.round(4))
print(f"test AUC 95% CI: {lo:.4f} to {hi:.4f}; test evaluation loss 95% CI: {c_lo:.4f} to {c_hi:.4f}")
print(f"test confusion at threshold {T_CAL:.3f}: TP {tp}, FN {fn}, FP {fp}, TN {tn} (recall {tp / (tp + fn):.1%}, precision {tp / (tp + fp):.1%}, tool holds {flag.mean():.1%} of wafers)")
print(f"for reference only (not used for any decision): test AUC of all four = "
      + ", ".join(f"{k[0]}/{k[1]} {roc_auc_score(yt, S_te[k]):.4f}" for k in MODELS))

summary = {"chosen_model": list(CHOSEN), "calibrator": CALIB, "threshold": round(T_CAL, 4), "cost_ratio_fn_fp": [C_FN, C_FP],
           "validation_auc": {f"{k[0]}|{k[1]}": round(float(roc_auc_score(yv, S_va[k])), 4) for k in MODELS},
           "test_auc_chosen": round(float(auc_te), 4), "test_eval_loss": round(float(cost_te), 4),
           "test_eval_loss_at_0.5": round(float(eval_loss(yt, p_te, 0.5)), 4), "configurations_tried": int(tuning["configurations tried"].sum())}
json.dump(summary, open("results_summary.json", "w"), indent=2); print(json.dumps(summary, indent=2))

,validation,test
AUC,0.9659,0.9584
evaluation loss at threshold 0.220,0.1429,0.1809
evaluation loss at threshold 0.5,0.1976,0.2179
ECE (calibrated),0.0212,0.0187
evaluation loss: hold every wafer,0.4089,0.4090
evaluation loss: never hold,2.9556,2.9551


test AUC 95% CI: 0.9463 to 0.9691; test evaluation loss 95% CI: 0.1431 to 0.2229
test confusion at threshold 0.220: TP 1011, FN 28, FP 178, TN 541 (recall 97.3%, precision 85.0%, tool holds 67.6% of wafers)
for reference only (not used for any decision): test AUC of all four = glassbox/L2 0.9528, glassbox/log loss 0.9586, blackbox/L2 0.9655, blackbox/log loss 0.9678
{
  "chosen_model": [
    "glassbox",
    "log loss"
  ],
  "calibrator": "isotonic",
  "threshold": 0.2204,
  "cost_ratio_fn_fp": [
    5.0,
    1.0
  ],
  "validation_auc": {
    "glassbox|L2": 0.9581,
    "glassbox|log loss": 0.9659,
    "blackbox|L2": 0.9655,
    "blackbox|log loss": 0.972
  },
  "test_auc_chosen": 0.9584,
  "test_eval_loss": 0.1809,
  "test_eval_loss_at_0.5": 0.2179,
  "configurations_tried": 24
}


## 11. Findings, and what to flag

### What we found
1. **The training loss matters, for both model types.** Log loss beat L2 loss on validation AUC for the glassbox (0.966 vs 0.958; difference +0.008, 95% CI +0.003 to +0.014) and for the blackbox (0.972 vs 0.966; +0.007, CI +0.003 to +0.011). The L2 models also give scores outside 0–1 for 20% (glassbox) and 29% (blackbox) of wafers, and the L2 glassbox is the worst calibrated of the four (ECE 0.107).
2. **We chose the glassbox (L1 logistic regression).** The blackbox has a slightly higher validation AUC (0.972 vs 0.966; +0.006, CI +0.0001 to +0.012). But at our cost ratio it doesn't lower the evaluation loss: 0.149 per wafer vs 0.143 for the glassbox. A higher AUC didn't mean a lower cost here, because AUC averages over all thresholds and our costs only care about the low-threshold end. By our rule, the blackbox didn't earn its opacity.
3. **The threshold is where the context pays off.** The cost-minimising threshold on the raw glassbox score is 0.28, not 0.5. That cuts the validation evaluation loss by 27% (0.196 → 0.143), by catching 31 more Edge-Loc wafers for 62 more unnecessary holds. Minimising plain misclassification would push the threshold the other way, to 0.61.
4. **Calibration.** The glassbox was mildly miscalibrated (ECE 0.031): it under-predicts in the 0.2–0.5 range, exactly where our threshold sits. Isotonic regression, fitted on out-of-fold training scores, brought ECE to about 0.02 without changing AUC. Platt scaling barely helped (0.029), which is expected: a logistic model's error isn't the S-shape that Platt corrects. For the blackbox, Platt works well (0.034 → 0.013).
5. **Calibration and threshold agree, roughly.** On calibrated probabilities the best threshold is 0.22; decision theory says 0.167. Using the formula's 0.167 would cost 0.148 against the searched 0.143, so the formula is a good default when costs change.
6. **The choice holds for cost ratios near ours.** The glassbox is at least as cheap as the blackbox for ratios 3 to 8. The blackbox is marginally cheaper at 1–2 and 9–10.
7. **Both models read the wafer the same way.** SHAP ranks `largest_cluster_radius` (where the biggest cluster sits) far ahead, then `edge_fail_rate` and `outer_ring_fail_rate`. That matches the physics of the question: is the cluster at the edge or not?
8. **The costliest mistake looks like a labelling question.** The Edge-Loc wafer the blackbox was most wrong about has its largest cluster near the centre (radius 0.32), and the glassbox agrees with the blackbox. Looking at the map, it's hard to see why it was labelled Edge-Loc.
9. **Test result (used once).** AUC 0.958 (95% CI 0.946 to 0.969), evaluation loss 0.181 per wafer (CI 0.143 to 0.223). That's 56% below holding the tool for every wafer (0.409) and 17% below using a 0.5 threshold (0.218).

### Flags: where this project departs from the brief, or is weaker than it looks
| # | Flag | What we did about it |
|---|---|---|
| 1 | **"Avoid large datasets."** The source is 811,457 maps in a 2 GB file. | We model 8,782 labelled wafers, saved as a 1 MB table. The notebook doesn't need the source file. |
| 2 | **"Avoid data cleaning."** The maps needed feature extraction (`build_features.py`). | This is deliberate: the features are our contribution. No rows were dropped or imputed. |
| 3 | **"High-usability Kaggle dataset."** We could not read the usability rating (Kaggle's API needs a login). | **Check the rating on the dataset's Kaggle page before presenting.** |
| 4 | **The 5:1 cost ratio is our assumption.** The dataset has no cost data. | Stated up front; section 8 shows ratios 1–10. |
| 5 | **The glassbox is less transparent than it sounds.** Cross-validation chose the weakest penalty in the grid (C = 10, the edge of the grid), which keeps all 10 features. Because the fail-rate features are correlated, the coefficients are large with opposite signs (`fail_rate` −22, `edge_fail_rate` +13), so they can't be read one at a time. | Shown openly in section 9. The penalty table in section 4 shows what a sparser model would cost in AUC. We did not re-tune after seeing the test split. |
| 6 | **Test is worse than validation for the chosen model.** Evaluation loss 0.181 vs 0.143; AUC 0.958 vs 0.966. | Expected in direction: the threshold and the model were both picked on validation (the optimiser's curse from the slides). The test figure is the honest one to quote. |
| 7 | **On test, the blackbox's AUC lead is larger** (0.968 vs 0.959) than on validation. | We report it and do not change the decision: choosing a model after seeing the test split would make the test result meaningless. |
| 8 | **Labels are engineers' judgements**, and some look debatable (finding 8). | Noted as a limit on any model's achievable accuracy. |
| 9 | **SHAP credit among the correlated fail-rate features is unstable.** | We interpret them as a group, and say so. |
| 10 | **Duplicates.** 55 wafers (0.6%) share feature values with another wafer; none have conflicting labels. | Kept. Lot-aware splitting limits any leakage from repeats within a lot. |
| 11 | **Our own screening notebook used random splits**, which ignore lots. | Fixed here. The effect is small (training-split CV AUC 0.974 random vs 0.972 by lot). |

### Good habits we followed
- 24 configurations in total, tuned inside the training split. Validation answered one four-way question and set one threshold. Test was used once.
- The decision rule was written down before the validation results.
- Every comparison has an interval from resampling whole lots.